# 14n -- Edge Pairs Probe (learned edge-classifier DE-RISK gate)

CPU-only probe for the Stage-4 learned edge-classifier (see
`docs/subagent-specs/edge-classifier-association.md` section 8).

Reuses the **frozen 14e/K7 features** (14h tracklets + primary CLIP-PCA +
DINOv2 tertiary; 14j R50-IBN quaternary), assigns each tracklet a GT
`global_id` via IoU majority vote, builds cross-camera labelled pairs in the
**exact live FIC+AQE feature space**, and runs a scene-disjoint LightGBM vs
the `cos_fused` threshold. Prints a GO/NO-GO verdict.

No GPU, no checkpoints written -- parquet pair tables + a JSON report only.

## 1. Imports + paths

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

import numpy as np

WORK_DIR = Path('/kaggle/working')
PROJECT = WORK_DIR / 'gp'
INPUT_ROOT = Path('/kaggle/input')
ASSEMBLED_RUN = Path('/tmp/edge_pairs_run')   # assembled stage1/stage2 run dir
OUT_DIR = Path('/kaggle/working/outputs/14n_edge_pairs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python: {sys.version.split()[0]}')
print(f'Kaggle input exists: {INPUT_ROOT.exists()}')

## 2. Clone repo + install CPU deps

In [ ]:
REPO_URL = 'https://github.com/MRKDaGods/gp.git'
REPO_BRANCH = 'paper-tests'   # branch carrying scripts/build_edge_pairs.py

if not PROJECT.exists():
    print(f'Cloning {REPO_URL} ({REPO_BRANCH}) ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, str(PROJECT)])
else:
    print('Repo present; pulling latest ...')
    subprocess.check_call(['git', '-C', str(PROJECT), 'pull', '--ff-only'])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))


def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


# build_edge_pairs imports src.stage4_association.* (numpy/loguru/omegaconf) +
# faiss is pulled in transitively by src.stage3_indexing; the probe itself
# needs lightgbm + pandas + pyarrow + scikit-learn.
pip('numpy', 'scipy', 'pandas', 'pyarrow', 'faiss-cpu', 'omegaconf', 'loguru',
    'networkx>=3.1', 'lightgbm', 'scikit-learn', 'pyyaml')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=str(PROJECT))
print(f'Repo ready at {PROJECT}')

In [ ]:
# Inlined build_edge_pairs.py (uncommitted -> absent from the cloned repo). Written here so the
# kernel runs without a git push; its src.stage4_association imports still resolve from the clone.
import base64, pathlib
_BEP_B64 = (
    "..."
    "KG1haW4oKSkK"
)
pathlib.Path('scripts').mkdir(exist_ok=True)
pathlib.Path('scripts/build_edge_pairs.py').write_bytes(base64.b64decode(_BEP_B64))
print('inlined scripts/build_edge_pairs.py:', pathlib.Path('scripts/build_edge_pairs.py').stat().st_size, 'bytes')

## 3. Resolve 14h anchor, 14j quaternary, and GT

Mirrors the 14k kernel's Cell 5 dataset-resolution logic. kernel_sources mount
under `/kaggle/input/notebooks/<owner>/<slug>/` (recurse-search as fallback).

In [ ]:
SOURCE_14H_OWNER_SLUG = 'yahiaakhalafallah/14h-robust-tracklet-pooling'
SOURCE_14J_OWNER_SLUG = 'yahiaakhalafallah/14j-r50-ibn-features'
SOURCE_14H_SLUG = SOURCE_14H_OWNER_SLUG.split('/', 1)[1]
SOURCE_14J_SLUG = SOURCE_14J_OWNER_SLUG.split('/', 1)[1]
EXPECTED_CAMS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
EXPECTED_TRACKLETS = 929


def find_input_dir(slug, owner_slug, hints=()):
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    owner, _, kernel = owner_slug.partition('/')
    nested = INPUT_ROOT / 'notebooks' / owner / kernel
    if nested.exists():
        return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(h).lower() for h in hints)
    for path in (list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []):
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(h in name for h in lowered_hints):
            return path
    return direct


def find_14h_checkpoint():
    source_dir = find_input_dir(SOURCE_14H_SLUG, SOURCE_14H_OWNER_SLUG, hints=('14h', 'robust', 'tracklet'))
    cp = source_dir / 'checkpoint.tar.gz'
    if cp.exists():
        print(f'14h input: {source_dir}')
        return cp
    visible = [str(p) for p in INPUT_ROOT.rglob('checkpoint.tar.gz')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f'14h checkpoint.tar.gz not found for {SOURCE_14H_OWNER_SLUG}. '
        f'Visible: {visible[:20]}')


checkpoint = find_14h_checkpoint()
EXTRACT_DIR = Path('/tmp/14h_checkpoint')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)')
with tarfile.open(str(checkpoint), 'r:gz') as archive:
    archive.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / 'run_metadata.json', encoding='utf-8') as fh:
    previous_meta = json.load(fh)
SOURCE_14H_RUN_NAME = previous_meta['run_name']
SOURCE_14H_RUN_DIR = EXTRACT_DIR / SOURCE_14H_RUN_NAME
SOURCE_STAGE1_DIR = SOURCE_14H_RUN_DIR / 'stage1'
SOURCE_STAGE2_DIR = SOURCE_14H_RUN_DIR / 'stage2'
for required in [
    SOURCE_STAGE1_DIR,
    SOURCE_STAGE2_DIR / 'embeddings.npy',
    SOURCE_STAGE2_DIR / 'embeddings_tertiary.npy',
    SOURCE_STAGE2_DIR / 'embedding_index.json',
]:
    if not required.exists():
        raise FileNotFoundError(required)
print(f'Loaded 14h run: {SOURCE_14H_RUN_NAME}')

In [ ]:
def find_quaternary_stage2_dir():
    source_dir = find_input_dir(SOURCE_14J_SLUG, SOURCE_14J_OWNER_SLUG, hints=('14j', 'r50', 'ibn'))
    candidates = [
        source_dir / 'outputs' / '14j_v4_features' / 'stage2',
        source_dir / '14j_v4_features' / 'stage2',
        source_dir / 'stage2',
    ]
    for cand in candidates:
        if (cand / 'embeddings_quaternary.npy').exists():
            print(f'14j quaternary input: {cand}')
            return cand
    matches = sorted(INPUT_ROOT.rglob('embeddings_quaternary.npy')) if INPUT_ROOT.exists() else []
    for m in matches:
        t = str(m).lower()
        if '14j' in t and ('r50' in t or 'ibn' in t or 'quaternary' in t):
            print(f'14j quaternary discovered: {m.parent}')
            return m.parent
    if matches:
        print(f'14j quaternary fallback: {matches[0].parent}')
        return matches[0].parent
    visible = [str(p) for p in INPUT_ROOT.rglob('*.npy')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f'embeddings_quaternary.npy not found for {SOURCE_14J_OWNER_SLUG}. '
        f'Visible npy: {visible[:30]}')


SOURCE_QUATERNARY_STAGE2_DIR = find_quaternary_stage2_dir()


def is_cityflow_gt_root(path):
    return path.exists() and all((path / cam / 'gt' / 'gt.txt').exists() for cam in EXPECTED_CAMS)


def find_cityflow_gt_root():
    candidates = [
        PROJECT / 'data' / 'raw' / 'cityflowv2',
        EXTRACT_DIR / 'gt_annotations',
        Path('/kaggle/input/data-aicity-2023-track-2'),
        Path('/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2'),
    ]
    for cand in candidates:
        if is_cityflow_gt_root(cand):
            return cand
    for gt_file in (INPUT_ROOT.rglob('gt.txt') if INPUT_ROOT.exists() else []):
        if gt_file.parent.name != 'gt' or gt_file.parent.parent.name not in EXPECTED_CAMS:
            continue
        cand = gt_file.parents[2]
        if is_cityflow_gt_root(cand):
            return cand
    visible = [str(p) for p in INPUT_ROOT.rglob('gt.txt')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        'CityFlowV2 GT not found in <root>/<cam>/gt/gt.txt layout. '
        f'Expected {EXPECTED_CAMS}. Visible gt.txt: {visible[:20]}')


GT_DIR = find_cityflow_gt_root()
print(f'Ground truth root: {GT_DIR}')

## 4. Assemble a single run dir (stage1 + 4-stream stage2)

`build_edge_pairs.py` expects one run dir with `stage1/` and `stage2/`
containing `embeddings.npy` (primary), `embeddings_tertiary.npy` (DINOv2),
`embeddings_quaternary.npy` (R50-IBN), and `embedding_index.json`. The 14h
checkpoint supplies stage1 + primary + tertiary; we copy the 14j quaternary
in, asserting its `embedding_index.json` matches the 14h ordering EXACTLY.

In [ ]:
# Validate quaternary index matches the 14h source ordering (row alignment).
src_index = json.loads((SOURCE_STAGE2_DIR / 'embedding_index.json').read_text(encoding='utf-8'))
quat_index = json.loads((SOURCE_QUATERNARY_STAGE2_DIR / 'embedding_index.json').read_text(encoding='utf-8'))
if src_index != quat_index:
    raise RuntimeError('14j quaternary embedding_index.json does not match 14h ordering')
if len(src_index) != EXPECTED_TRACKLETS:
    raise RuntimeError(f'Expected {EXPECTED_TRACKLETS} rows, found {len(src_index)}')

quat = np.load(SOURCE_QUATERNARY_STAGE2_DIR / 'embeddings_quaternary.npy').astype(np.float32)
if quat.shape[0] != EXPECTED_TRACKLETS:
    raise RuntimeError(f'Unexpected quaternary shape: {quat.shape}')
print(f'Quaternary embeddings: {quat.shape}, finite={np.isfinite(quat).all()}')

# Assemble the run dir.
if ASSEMBLED_RUN.exists():
    shutil.rmtree(ASSEMBLED_RUN)
(ASSEMBLED_RUN / 'stage2').mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE_STAGE1_DIR, ASSEMBLED_RUN / 'stage1')
for fname in ['embeddings.npy', 'embeddings_tertiary.npy', 'embedding_index.json']:
    shutil.copy2(SOURCE_STAGE2_DIR / fname, ASSEMBLED_RUN / 'stage2' / fname)
# hsv_features.npy is optional for the probe; copy it if present for completeness.
if (SOURCE_STAGE2_DIR / 'hsv_features.npy').exists():
    shutil.copy2(SOURCE_STAGE2_DIR / 'hsv_features.npy', ASSEMBLED_RUN / 'stage2' / 'hsv_features.npy')
np.save(ASSEMBLED_RUN / 'stage2' / 'embeddings_quaternary.npy', quat)

print('Assembled run dir contents:')
for p in sorted((ASSEMBLED_RUN / 'stage2').iterdir()):
    print(f'  stage2/{p.name}')
print(f'  stage1/: {len(list((ASSEMBLED_RUN / "stage1").glob("tracklets_*.json")))} tracklet files')

## 5. (sanity) Self-test the probe before the real run

Exercises every code path on tiny synthetic data -- catches an env/dependency
break before we spend time on the real features.

In [ ]:
rc = subprocess.call([sys.executable, 'scripts/build_edge_pairs.py', '--self-test'])
print(f'self-test exit code: {rc}')
if rc != 0:
    raise RuntimeError('build_edge_pairs self-test failed -- fix env before the real run')

## 6. Run the probe on the real frozen features

FIC+AQE feature space (matches the live K7 gate), K7 fusion weights read from
`configs/model_registry.yaml`. Emits `edge_pairs_S01.parquet` /
`edge_pairs_S02.parquet` + `separability_report.json` and prints the verdict.

In [ ]:
cmd = [
    sys.executable, 'scripts/build_edge_pairs.py',
    '--run-dir', str(ASSEMBLED_RUN),
    '--gt-root', str(GT_DIR),
    '--out-dir', str(OUT_DIR),
]
print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
# ascii-sanitize before printing (Kaggle stdout is cp1252-safe).
print(proc.stdout.encode('ascii', 'replace').decode('ascii'))
if proc.stderr.strip():
    print('--- stderr (tail) ---')
    print('\n'.join(proc.stderr.encode('ascii', 'replace').decode('ascii').splitlines()[-40:]))
if proc.returncode != 0:
    raise RuntimeError(f'build_edge_pairs.py failed with exit {proc.returncode}')

## 7. Surface the verdict + outputs

In [ ]:
report_path = OUT_DIR / 'separability_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))

print('=' * 70)
print('14n EDGE-PAIRS PROBE -- VERDICT')
print('=' * 70)
print(f"verdict                : {report.get('verdict')}")
print(f"mean model AUC (hard)  : {report.get('mean_model_auc_hardneg')}")
print(f"mean baseline cos_fused: {report.get('mean_baseline_auc_hardneg')}")
print(f"mean delta             : {report.get('mean_delta')}")
print(f"K7 weights             : {report.get('k7_weights')}")
for fold in report.get('folds', []):
    print(f"  fold held-out {fold['held_out_scene']}: "
          f"model_all={fold['model_auc_all']:.4f} base_all={fold['baseline_auc_all']:.4f} "
          f"delta_hard={fold.get('delta_hardneg')}")

print('\nOutput files:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p}  ({p.stat().st_size} bytes)')